In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from dotenv import load_dotenv

# Disable symlinks warning in datasets
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [3]:
DS_SIZE = 1000 # 1000 bald, 1000 not-bald for each split (total 4000)
IMAGES_DIR = "data/images"
METADATA_PATH = "data/metadata.csv"

os.makedirs(IMAGES_DIR, exist_ok=True)

In [4]:
def process_stream(stream, target_count, split_name, current_idx, metadata_list):
    bald_count = 0
    not_bald_count = 0
    
    print(f"Collecting {target_count} bald and {target_count} not-bald for {split_name}...")
    for item in tqdm(stream):
        is_bald = item['Bald']
        
        keep = False
        if is_bald is True and bald_count < target_count:
            bald_count += 1
            keep = True
        elif is_bald is False and not_bald_count < target_count:
            not_bald_count += 1
            keep = True
            
        if keep:
            img = item['image'].convert("RGB")
            # Resize to ViT standard size to save space
            img = img.resize((224, 224))
            filename = f"img_{current_idx:05d}.jpg"
            img_path = os.path.join(IMAGES_DIR, filename)
            img.save(img_path, quality=90)
            
            metadata_list.append({
                'filename': filename,
                'bald': int(is_bald),
                'split': split_name
            })
            current_idx += 1
            
        if bald_count >= target_count and not_bald_count >= target_count:
            break
            
    return current_idx

In [5]:
if os.path.exists(METADATA_PATH):
    print("Metadata already exists. Please delete if you want to re-download.")
else:
    print("Streaming data from HuggingFace...")
    ds = load_dataset("flwrlabs/celeba", split="train", streaming=True, token=HF_TOKEN)
    ds_iterator = iter(ds)
    
    metadata_list = []
    idx = 0
    
    # Collect train (set A for CAV)
    idx = process_stream(ds_iterator, DS_SIZE, "train", idx, metadata_list)
    
    # Collect test (set B for Recovery Analysis)
    idx = process_stream(ds_iterator, DS_SIZE, "test", idx, metadata_list)
    
    # Save metadata
    df = pd.DataFrame(metadata_list)
    df.to_csv(METADATA_PATH, index=False)
    print(f"Saved metadata with {len(df)} records.")

Streaming data from HuggingFace...


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

41675it [07:04, 98.18it/s] 


44870it [07:17, 102.59it/s]


Saved metadata with 4000 records.
